# **Data Preprocessing**

In [ ]:
# conda activate "F:\_GitHub\CSCI161-SocialComputing\Binwag - PhilippineRedditAgricultureAnalysis\env"
# OR go into the directory above, and conda activate
# conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia

# conda activate ./env
# jupyter notebook


In [ ]:
import os
import sys
import subprocess

print("Checking GPU and environment...")

conda_env = os.environ.get("CONDA_PREFIX", None)
print(f"Conda environment: {conda_env if conda_env else 'Not in Conda'}")

try:
    result = subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if result.returncode == 0:
        print("NVIDIA GPU detected.")
        print(result.stdout.split("\n")[2]) 
    else:
        print("No NVIDIA GPU found or drivers missing.")
except FileNotFoundError:
    print("'nvidia-smi' not found — NVIDIA drivers may not be installed.")

try:
    import torch
    print("PyTorch already installed.")
except ImportError:
    print("Installing PyTorch with CUDA 12.1 support...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "torch", "torchvision", "torchaudio",
        "--extra-index-url", "https://download.pytorch.org/whl/cu121"
    ])
    import torch

try:
    import torch
    print("PyTorch CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("Using GPU:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available — running on CPU.")
except Exception as e:
    print("PyTorch check failed:", e)
    print("Trying TensorFlow GPU check instead...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow[and-cuda]"])
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    print("TensorFlow GPUs detected:" if gpus else "TensorFlow found no GPU.", gpus)

print("Setup complete.")


In [ ]:
import pandas as pd
import os

print(f"Current directory: {os.getcwd()}")

reddit_raw = pd.read_csv(r"data/Reddit/reddit_data.csv")
rappler_raw = pd.read_csv(r"data/News/rappler_agriculture.csv")

reddit_raw.info()
reddit_raw.tail()



In [ ]:
rappler_raw.info()
rappler_raw.tail()

---
**Cleaning Reddit Data**

In [ ]:
import pandas as pd
import re

reddit_clean = reddit_raw.drop(columns=['comment_author', 'post_author','post_score','comment_score'])

reddit_clean = reddit_clean.dropna(subset=['post_title', 'comment'], how='all')

# Combine title + body into one field (if body exists)
reddit_clean['post_full'] = reddit_clean.apply(
    lambda row: f"{row['post_title']} {row['post_body']}" if pd.notnull(row['post_body']) else row['post_title'],
    axis=1
)

# Text normalization
def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"http\S+|www\S+", "", text)       # remove URLs
    text = re.sub(r"<.*?>", "", text)                # remove HTML tags
    text = re.sub(r"[\r\n]+", " ", text)             # newlines → space
    text = re.sub(r"\s+", " ", text).strip()         # normalize whitespace
    text = text.lower()                              # lowercase
    text = text.replace("[deleted]", "").replace("[removed]", "")
    return text.strip()

reddit_clean['post_full'] = reddit_clean['post_full'].apply(clean_text)
reddit_clean['comment'] = reddit_clean['comment'].apply(clean_text)

reddit_clean = reddit_clean[(reddit_clean['post_full'].str.len() > 0) | (reddit_clean['comment'].str.len() > 0)]

reddit_clean = reddit_clean.drop_duplicates(subset=['post_url', 'comment'])

reddit_clean = reddit_clean.reset_index(drop=True)

reddit_clean.to_csv("data/Reddit/reddit_clean.csv", index=False, encoding='utf-8-sig')

print(f"Cleaned {len(reddit_clean)} rows saved to reddit_clean.csv")
reddit_clean.tail(3)


Reddit is cleaned by normalizing text, duplicates, and removing unnused columns for this study.

In [ ]:
rappler_clean = rappler_raw.copy()

# Drop the 'author' column
rappler_clean = rappler_clean.drop(columns=['author'])

# Text normalization function
def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"http\S+|www\S+", "", text)       # remove URLs
    text = re.sub(r"<.*?>", "", text)                # remove HTML tags
    text = re.sub(r"[\r\n]+", " ", text)            # newlines → space
    text = re.sub(r"\s+", " ", text).strip()        # normalize whitespace
    text = text.lower()                              # lowercase
    text = text.replace("[deleted]", "").replace("[removed]", "")
    return text.strip()

rappler_clean['cleaned_text'] = rappler_clean['text'].apply(clean_text)
rappler_clean.to_csv(r"data/News/rappler_clean.csv", index=False)

rappler_clean.tail()

---

Final overview

In [ ]:
rappler_clean.info()
rappler_clean.isna().sum()


In [ ]:
rappler_clean[['text', 'cleaned_text']].tail()
